In [6]:
import pickle
import os
import networkx as nx
import numpy as np

DATA_PATH = '../../pickle_files/data_30-01-2026_16-24-39/'
GRAPH_PATH = DATA_PATH + '/graph.pkl'
GROUND_TRUTH_PATH = DATA_PATH + '/ground_truth.pkl'
MEASUREMENTS_PATH = DATA_PATH + '/measurements.pkl'
OUTPUT_PATH = DATA_PATH + '/single_edge_data/'

class TimeAwareMeasurement:
    def __init__(self, path_edges, histogram, duration, latency_stats=None, code_type='513'):
        """
        :param path_edges: List of tuples representing the path, e.g., [(0,1), (1,2)]
        :param histogram: A 16-dimensional numpy array of syndrome counts.
        :param duration: The duration or average latency of this measurement (seconds).
        :param latency_stats: A dictionary with 'max', 'std', 'count'.
        :param code_type: String indicating the code type, e.g., '313_X', '313_Z', '513'.
        """
        self.path_edges = path_edges
        self.histogram = histogram
        self.duration = duration
        self.latency_stats = latency_stats
        self.code_type = code_type  # Default, will be updated per sample


with open(GRAPH_PATH, 'rb') as f:
    graph = pickle.load(f)
with open(GROUND_TRUTH_PATH, 'rb') as f:
    ground_truth = pickle.load(f)
with open(MEASUREMENTS_PATH, 'rb') as f:
    measurements = pickle.load(f)


In [7]:
print(graph.edges(data=True))
print(graph.nodes(data=True))

[(0, 1, {}), (0, 3, {}), (1, 2, {}), (1, 4, {}), (3, 4, {}), (2, 5, {}), (4, 5, {})]
[(0, {}), (1, {}), (3, {}), (2, {}), (4, {}), (5, {})]


In [8]:
for measure in measurements:
    print(measure.path_edges, measure.code_type)

[(0, 1)] 313_X
[(0, 1)] 313_Z
[(0, 3)] 313_X
[(0, 3)] 313_Z
[(1, 0)] 313_X
[(1, 0)] 313_Z
[(1, 2)] 313_X
[(1, 2)] 313_Z
[(1, 4)] 313_X
[(1, 4)] 313_Z
[(2, 1)] 313_X
[(2, 1)] 313_Z
[(2, 5)] 313_X
[(2, 5)] 313_Z
[(3, 0)] 313_X
[(3, 0)] 313_Z
[(3, 4)] 313_X
[(3, 4)] 313_Z
[(4, 1)] 313_X
[(4, 1)] 313_Z
[(4, 3)] 313_X
[(4, 3)] 313_Z
[(4, 5)] 313_X
[(4, 5)] 313_Z
[(5, 2)] 313_X
[(5, 2)] 313_Z
[(5, 4)] 313_X
[(5, 4)] 313_Z


In [9]:
for truth in ground_truth:
    print(truth)

(0, 1)
(0, 3)
(1, 2)
(1, 4)
(2, 5)
(3, 4)
(4, 5)


In [10]:
for edge in graph.edges(data=True):
    u, v, attr = edge

    path_edges = [(u, v)]
    path_edges_rev = [(v, u)]
    measurement_subset = []
    
    for sample in measurements:
        if sample.path_edges != path_edges and sample.path_edges != path_edges_rev:
            continue  # Skip samples that don't match the current edge

        measurement_subset.append(sample)

    print(f"Processing edge: {u}-{v} with {len(measurement_subset)} samples")
    new_graph = nx.Graph()
    new_graph.add_edges_from(path_edges)
    new_ground_truth = {(u,v):ground_truth[(u,v)]} if (u,v) in ground_truth else {(v,u):ground_truth[(v,u)]}

    edge_path = f"{u}_{v}"

    os.makedirs(f"{OUTPUT_PATH}/{edge_path}", exist_ok=True)

    with open(f"{OUTPUT_PATH}/{edge_path}/graph.pkl", "wb") as f:
        pickle.dump(new_graph, f)

    with open(f"{OUTPUT_PATH}/{edge_path}/ground_truth.pkl", "wb") as f:
        pickle.dump(new_ground_truth, f)

    with open(f"{OUTPUT_PATH}/{edge_path}/measurements.pkl", "wb") as f:
        pickle.dump(measurement_subset, f)
    

Processing edge: 0-1 with 4 samples
Processing edge: 0-3 with 4 samples
Processing edge: 1-2 with 4 samples
Processing edge: 1-4 with 4 samples
Processing edge: 3-4 with 4 samples
Processing edge: 2-5 with 4 samples
Processing edge: 4-5 with 4 samples
